# Diabetes Complication Risk Predictor

**Dataset:** UCI Diabetes 130-US Hospitals (1999–2008)  
**Model:** XGBoost + LightGBM + Random Forest — Soft Voting Ensemble  
**Target:** Early hospital readmission risk (binary: 0 = Stable, 1 = High-Risk)

---

## Contents

1. [Setup & Imports](#1)
2. [Load Dataset](#2)
3. [Exploratory Data Analysis](#3)
4. [Data Cleaning & Preprocessing](#4)
5. [Feature Engineering](#5)
6. [Train/Test Split & SMOTE](#6)
7. [Feature Scaling](#7)
8. [Model Training](#8)
9. [Evaluation](#9)
10. [SHAP Explainability](#10)
11. [Save Artifacts](#11)
12. [Prediction Test](#12)

<a id='1'></a>
## 1. Setup & Imports

Run this cell once to install all dependencies, then restart the kernel before proceeding.

In [ ]:
import subprocess, sys

packages = [
    "pandas>=2.0.0",
    "numpy>=1.24.0",
    "scikit-learn>=1.3.0",
    "xgboost>=2.0.0",
    "lightgbm>=4.0.0",
    "imbalanced-learn>=0.11.0",
    "shap>=0.44.0",
    "joblib>=1.3.0",
    "matplotlib>=3.7.0",
    "seaborn>=0.12.0",
    "ucimlrepo>=0.0.3",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("Installation complete. Restart the kernel before running the next cells.")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import io
import time
import json
import zipfile
import joblib
import shap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8fafc',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports OK")

<a id='2'></a>
## 2. Load Dataset

If `diabetic_data.csv` is already in this folder (>100 KB), it will be used directly.  
Otherwise the cell downloads it automatically via two fallback methods.

In [ ]:
import urllib.request

CSV_PATH = "diabetic_data.csv"
ZIP_URL  = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"

def _is_valid_csv(path, min_bytes=100_000):
    return os.path.exists(path) and os.path.getsize(path) >= min_bytes

def _download_via_ucimlrepo():
    from ucimlrepo import fetch_ucirepo
    print("  Trying ucimlrepo...")
    ds = fetch_ucirepo(id=296)
    df = ds.data.features.copy()
    tgt = ds.data.targets
    # targets may be DataFrame (2-D) or Series — normalise to 1-D
    if hasattr(tgt, 'iloc'):
        tgt = tgt.iloc[:, 0]
    df['readmitted'] = tgt.values
    df.to_csv(CSV_PATH, index=False)
    return df.shape

def _download_via_zip():
    print("  Trying direct ZIP download...")
    with urllib.request.urlopen(ZIP_URL, timeout=120) as resp:
        data = resp.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        names = zf.namelist()
        csv_names = [n for n in names if 'diabetic_data' in n.lower() and n.endswith('.csv')]
        if not csv_names:
            csv_names = [n for n in names if n.endswith('.csv')]
        target = csv_names[0]
        print(f"  Extracting: {target}")
        with zf.open(target) as src, open(CSV_PATH, 'wb') as dst:
            dst.write(src.read())
    df = pd.read_csv(CSV_PATH, low_memory=False)
    return df.shape

if _is_valid_csv(CSV_PATH):
    print(f"Found: {CSV_PATH}  ({os.path.getsize(CSV_PATH)/1024/1024:.1f} MB)")
else:
    if os.path.exists(CSV_PATH):
        os.remove(CSV_PATH)
        print("Removed corrupt/empty file. Downloading fresh copy...")
    else:
        print("File not found. Downloading...")

    success = False
    for name, fn in [('ucimlrepo', _download_via_ucimlrepo),
                     ('direct ZIP', _download_via_zip)]:
        try:
            shape = fn()
            print(f"  Saved {CSV_PATH}: {shape[0]:,} rows x {shape[1]} columns")
            success = True
            break
        except Exception as e:
            print(f"  {name} failed: {e}")

    if not success:
        raise RuntimeError(
            "Both download methods failed.\n"
            "Download manually from: https://archive.ics.uci.edu/dataset/296/\n"
            "Unzip and place diabetic_data.csv in this folder, then re-run this cell."
        )

In [ ]:
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Shape  : {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")

In [ ]:
df_raw.head()

In [ ]:
df_raw.info()

<a id='3'></a>
## 3. Exploratory Data Analysis

In [ ]:
readmit_counts = df_raw['readmitted'].value_counts()
print(readmit_counts)
print((readmit_counts / len(df_raw) * 100).round(2).astype(str) + '%')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = ['#22c55e', '#f59e0b', '#ef4444']
axes[0].bar(readmit_counts.index, readmit_counts.values, color=colors, width=0.5, edgecolor='white')
axes[0].set_title('Readmission — 3 Classes', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Readmission Category')
axes[0].set_ylabel('Patient Count')
for i, (cat, val) in enumerate(readmit_counts.items()):
    axes[0].text(i, val + 200, f'{val:,}\n({val/len(df_raw)*100:.1f}%)', ha='center', fontsize=10)

binary_target = (df_raw['readmitted'] == '<30').astype(int)
binary_counts = binary_target.value_counts()
axes[1].bar(['Stable (0)', 'High-Risk (1)'], binary_counts.values,
            color=['#22c55e', '#ef4444'], width=0.45, edgecolor='white')
axes[1].set_title('Binary Target (after mapping)', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Patient Count')
for i, val in enumerate(binary_counts.values):
    axes[1].text(i, val + 200, f'{val:,}\n({val/len(df_raw)*100:.1f}%)', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Missing values — UCI-130 uses '?' as placeholder
df_miss = df_raw.replace('?', np.nan)
missing = df_miss.isna().sum()
missing_pct = (missing / len(df_miss) * 100).round(2)
miss_df = pd.DataFrame({'Count': missing, 'Pct': missing_pct})
miss_df = miss_df[miss_df['Count'] > 0].sort_values('Pct', ascending=False)
print(miss_df.to_string())

if len(miss_df) > 0:
    fig, ax = plt.subplots(figsize=(10, max(3, len(miss_df) * 0.45)))
    bars = ax.barh(miss_df.index, miss_df['Pct'],
                   color=['#ef4444' if p > 50 else '#f59e0b' if p > 20 else '#3b82f6'
                          for p in miss_df['Pct']])
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Value Rate by Column', fontweight='bold')
    ax.axvline(50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
    ax.legend()
    for bar, pct in zip(bars, miss_df['Pct']):
        ax.text(pct + 0.5, bar.get_y() + bar.get_height()/2, f'{pct:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('eda_missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
age_order = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
             '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']
age_counts = df_raw['age'].value_counts().reindex(age_order, fill_value=0)
axes[0].bar(age_counts.index, age_counts.values, color='#3b82f6', edgecolor='white')
axes[0].set_title('Patient Age Distribution', fontweight='bold')
axes[0].set_xlabel('Age Bracket')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

gender_counts = df_raw['gender'].value_counts()
axes[1].pie(gender_counts.values, labels=gender_counts.index,
            autopct='%1.1f%%', colors=['#3b82f6', '#ec4899', '#94a3b8'],
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Gender Distribution', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
num_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
            'num_medications', 'number_outpatient', 'number_emergency',
            'number_inpatient', 'number_diagnoses']
num_cols = [c for c in num_cols if c in df_raw.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df_raw[col].dropna(), bins=30, color='#6366f1', edgecolor='white', alpha=0.85)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='4'></a>
## 4. Data Cleaning & Preprocessing

In [ ]:
df = df_raw.copy()
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
df = df.replace('?', np.nan)
print(f"Initial shape: {df.shape}")

In [ ]:
# Drop columns: IDs (no signal), high missingness, near-zero variance
DROP_COLS = [
    'encounter_id', 'patient_nbr',
    'weight', 'payer_code', 'medical_specialty',
    'examide', 'citoglipton',
]
dropped = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=dropped)
print(f"Dropped: {dropped}  |  Shape: {df.shape}")

In [ ]:
# Binary target: readmitted within 30 days = 1, otherwise = 0
df['target'] = (df['readmitted'] == '<30').astype(int)
df = df.drop(columns=['readmitted'])

class_counts = df['target'].value_counts()
print(f"Stable (0)   : {class_counts[0]:>7,}  ({class_counts[0]/len(df)*100:.1f}%)")
print(f"High-Risk (1): {class_counts[1]:>7,}  ({class_counts[1]/len(df)*100:.1f}%)")

In [ ]:
# Ordinal encode lab results
a1c_map = {'>8': 3, '>7': 2, 'Norm': 1, 'None': 0, np.nan: 0}
if 'a1cresult' in df.columns:
    df['a1cresult'] = df['a1cresult'].map(a1c_map).fillna(0).astype(int)

glu_map = {'>300': 3, '>200': 2, 'Norm': 1, 'None': 0, np.nan: 0}
if 'max_glu_serum' in df.columns:
    df['max_glu_serum'] = df['max_glu_serum'].map(glu_map).fillna(0).astype(int)

In [ ]:
# Map ICD-9 diagnosis codes to disease chapter integers (0-17)
def group_icd9(code) -> int:
    if pd.isna(code):
        return 0
    code_str = str(code).strip().upper()
    if code_str.startswith('V') or code_str.startswith('E'):
        return 8
    try:
        num = float(code_str)
    except ValueError:
        return 0
    if   1   <= num <= 139:  return 1
    elif 140 <= num <= 239:  return 2
    elif 240 <= num <= 279:  return 3
    elif 280 <= num <= 289:  return 4
    elif 290 <= num <= 319:  return 5
    elif 320 <= num <= 389:  return 6
    elif 390 <= num <= 459:  return 7
    elif 460 <= num <= 519:  return 8
    elif 520 <= num <= 579:  return 9
    elif 580 <= num <= 629:  return 10
    elif 630 <= num <= 679:  return 11
    elif 680 <= num <= 709:  return 12
    elif 710 <= num <= 739:  return 13
    elif 740 <= num <= 759:  return 14
    elif 760 <= num <= 779:  return 15
    elif 780 <= num <= 799:  return 16
    elif 800 <= num <= 999:  return 17
    return 0

for diag_col in ['diag_1', 'diag_2', 'diag_3']:
    if diag_col in df.columns:
        df[diag_col] = df[diag_col].apply(group_icd9)
print("ICD-9 codes encoded")

In [ ]:
# Encode remaining categorical features
if 'gender' in df.columns:
    df['gender'] = df['gender'].map({'Male': 1, 'Female': 0, 'Unknown/Invalid': np.nan})

age_map = {
    '[0-10)': 5,   '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}
if 'age' in df.columns:
    df['age'] = df['age'].map(age_map)

if 'change' in df.columns:
    df['change'] = df['change'].map({'Ch': 1, 'No': 0})

if 'diabetesmed' in df.columns:
    df['diabetesmed'] = df['diabetesmed'].map({'Yes': 1, 'No': 0})

med_dosage_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide',
    'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone', 'insulin'
]
med_map = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': -1}
for col in med_dosage_cols:
    if col in df.columns:
        df[col] = df[col].map(med_map).fillna(0).astype(int)

obj_cols = [c for c in df.select_dtypes(include='object').columns if c != 'target']
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, drop_first=True, dtype=int)
    print(f"One-hot encoded: {obj_cols}")

print(f"Shape after encoding: {df.shape}")

In [ ]:
# Impute NaN with column median, then drop any remaining non-numeric columns
before = df.isna().sum().sum()
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isna().sum() > 0:
        df[col] = df[col].fillna(df[col].median())
after = df.isna().sum().sum()
print(f"Missing before: {before:,} | After: {after:,}")

df = df.select_dtypes(include=[np.number])
print(f"Final shape: {df.shape}")
df.describe().T

<a id='5'></a>
## 5. Feature Engineering

In [ ]:
if 'num_medications' in df.columns and 'num_procedures' in df.columns:
    df['med_procedure_ratio'] = df['num_medications'] / (df['num_procedures'] + 1)

if 'number_inpatient' in df.columns and 'number_emergency' in df.columns:
    df['hospital_complexity'] = df['number_inpatient'] + df['number_emergency'] * 2

if 'number_outpatient' in df.columns and 'number_inpatient' in df.columns:
    df['total_prior_visits'] = df['number_outpatient'].fillna(0) + df['number_inpatient'].fillna(0)

if 'diag_1' in df.columns and 'diag_2' in df.columns:
    df['multi_diag_endocrine'] = (df['diag_1'] == 3).astype(int) + (df['diag_2'] == 3).astype(int)

if 'admission_source_id' in df.columns:
    df['is_emergency_admission'] = (df['admission_source_id'] == 7).astype(int)

if 'time_in_hospital' in df.columns:
    df['long_stay'] = (df['time_in_hospital'] > 7).astype(int)

feature_cols = [c for c in df.columns if c != 'target']
print(f"Total features: {len(feature_cols)}")

In [ ]:
corr = df[feature_cols + ['target']].corr()['target'].drop('target').sort_values(key=abs, ascending=False)
top_n = min(25, len(corr))
top_corr = corr.head(top_n)

fig, ax = plt.subplots(figsize=(10, max(5, top_n * 0.38)))
colors = ['#ef4444' if v > 0 else '#3b82f6' for v in top_corr.values]
ax.barh(top_corr.index[::-1], top_corr.values[::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Target')
ax.set_title(f'Top {top_n} Feature Correlations', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print(corr.head(10).to_string())

<a id='6'></a>
## 6. Train/Test Split & SMOTE

In [ ]:
X = df[feature_cols].values
y = df['target'].values
print(f"X: {X.shape}  |  y: {y.shape}  |  Positive rate: {y.mean()*100:.2f}%")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {len(X_train):,}  (Neg={sum(y_train==0):,} | Pos={sum(y_train==1):,})")
print(f"After  SMOTE: {len(X_train_bal):,}  (Neg={sum(y_train_bal==0):,} | Pos={sum(y_train_bal==1):,})")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, counts, title in [
    (axes[0], [sum(y_train==0), sum(y_train==1)], 'Before SMOTE'),
    (axes[1], [sum(y_train_bal==0), sum(y_train_bal==1)], 'After SMOTE'),
]:
    ax.bar(['Stable (0)', 'High-Risk (1)'], counts, color=['#22c55e', '#ef4444'], width=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts):
        ax.text(i, v + 100, f'{v:,}', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('smote_balance.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='7'></a>
## 7. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_bal)
X_test_s  = scaler.transform(X_test)
print(f"Train: {X_train_s.shape}  |  Test: {X_test_s.shape}")

<a id='8'></a>
## 8. Model Training

Soft-voting ensemble: XGBoost (weight 2) + LightGBM (weight 2) + Random Forest (weight 1).

> **This cell takes 5–10 minutes. Wait for it to print "Training complete" before running any later cells.**

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0, n_jobs=-1,
)

lgbm_model = LGBMClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=RANDOM_STATE, verbose=-1, n_jobs=-1,
)

rf_model = RandomForestClassifier(
    n_estimators=250, max_depth=10, min_samples_split=10,
    min_samples_leaf=5, max_features='sqrt',
    random_state=RANDOM_STATE, n_jobs=-1,
)

ensemble = VotingClassifier(
    estimators=[('xgb', xgb_model), ('lgbm', lgbm_model), ('rf', rf_model)],
    voting='soft',
    weights=[2, 2, 1],
)

print("Training ensemble — please wait...")
t_start = time.time()
ensemble.fit(X_train_s, y_train_bal)
t_elapsed = time.time() - t_start
print(f"Training complete — {t_elapsed/60:.1f} min ({t_elapsed:.0f} sec)")

# Verify the model is fitted before proceeding
check_is_fitted(ensemble)
print("Model fitted successfully. You can now run the evaluation cells.")

<a id='9'></a>
## 9. Evaluation

In [ ]:
# Guard: ensure model is trained before evaluating
try:
    check_is_fitted(ensemble)
except Exception:
    raise RuntimeError(
        "The ensemble model is not trained yet.\n"
        "Run Section 8 (Model Training) first and wait for it to print 'Training complete'."
    )

y_pred      = ensemble.predict(X_test_s)
y_pred_prob = ensemble.predict_proba(X_test_s)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc_roc  = roc_auc_score(y_test, y_pred_prob)
cm       = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv         = tp / (tp + fp)

print(f"Accuracy    : {accuracy*100:.2f}%")
print(f"AUC-ROC     : {auc_roc:.4f}")
print(f"Sensitivity : {sensitivity*100:.2f}%")
print(f"Specificity : {specificity*100:.2f}%")
print(f"Precision   : {ppv*100:.2f}%")
print(f"\nTN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")
print()
print(classification_report(y_test, y_pred, target_names=['Stable (0)', 'High-Risk (1)']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Stable', 'High-Risk'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=12)

fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc_val = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#3b82f6', lw=2.5, label=f'AUC = {roc_auc_val:.4f}')
axes[1].plot([0, 1], [0, 1], color='#94a3b8', linestyle='--', lw=1.5)
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.02])
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve', fontweight='bold', fontsize=12)
axes[1].legend(loc='lower right', fontsize=10)
axes[1].fill_between(fpr, tpr, alpha=0.08, color='#3b82f6')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5-fold cross-validation on balanced training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(
    ensemble, X_train_s, y_train_bal, cv=cv, scoring='roc_auc', n_jobs=-1
)
print(f"Fold AUC-ROC: {[f'{s:.4f}' for s in cv_scores]}")
print(f"Mean        : {cv_scores.mean():.4f}")
print(f"Std         : {cv_scores.std():.4f}")
print(f"95% CI      : [{cv_scores.mean()-2*cv_scores.std():.4f}, {cv_scores.mean()+2*cv_scores.std():.4f}]")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(y_pred_prob[y_test == 0], bins=50, alpha=0.65, color='#22c55e', label='Stable', density=True)
ax.hist(y_pred_prob[y_test == 1], bins=50, alpha=0.65, color='#ef4444', label='High-Risk', density=True)
ax.axvline(0.5, color='black', linestyle='--', lw=1.5, label='Threshold = 0.5')
ax.set_xlabel('Predicted Probability of High-Risk', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Prediction Score Distribution by True Class', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('probability_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='10'></a>
## 10. SHAP Explainability

VotingClassifier is not directly supported by SHAP TreeExplainer, so a standalone XGBoost model with the same hyperparameters is trained for SHAP computation.

In [ ]:
xgb_shap = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0, n_jobs=-1,
)
xgb_shap.fit(X_train_s, y_train_bal)
print("XGBoost (SHAP) training complete")

In [ ]:
shap_sample_size = min(2000, len(X_test_s))
np.random.seed(RANDOM_STATE)
shap_idx = np.random.choice(len(X_test_s), size=shap_sample_size, replace=False)
X_shap   = X_test_s[shap_idx]

explainer   = shap.TreeExplainer(xgb_shap)
shap_values = explainer.shap_values(X_shap)
print(f"SHAP values shape: {np.array(shap_values).shape}")

In [ ]:
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_cols, show=False, plot_type='bar', max_display=20)
plt.title('Feature Importance — SHAP Mean Absolute Values', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_feature_importance_v3.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_cols, show=False, max_display=15)
plt.title('SHAP Beeswarm — Feature Direction and Magnitude', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_beeswarm_v3.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
y_shap_true = y_test[shap_idx]
high_risk_indices = np.where(y_shap_true == 1)[0]

if len(high_risk_indices) > 0:
    patient_idx = high_risk_indices[0]
    shap_explanation = explainer(X_shap)
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(shap_explanation[patient_idx], max_display=12, show=False)
    plt.title(f'SHAP Waterfall — High-Risk Patient (index {patient_idx})',
              fontsize=12, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig('shap_waterfall_v3.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print("No high-risk patients in sample — increase shap_sample_size.")

<a id='11'></a>
## 11. Save Artifacts

In [ ]:
# Guard: ensure model is fitted before saving
try:
    check_is_fitted(ensemble)
except Exception:
    raise RuntimeError(
        "Cannot save — ensemble is not fitted.\n"
        "Run Section 8 (Model Training) first."
    )

joblib.dump(ensemble,  'diabetes_model_v3.pkl')
joblib.dump(scaler,    'scaler_v3.pkl')
joblib.dump(explainer, 'shap_explainer_v3.pkl')

with open('feature_names_v3.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)

model_size = os.path.getsize('diabetes_model_v3.pkl') / 1024**2
shap_size  = os.path.getsize('shap_explainer_v3.pkl') / 1024**2

# Validate the saved file is not empty
if model_size < 0.1:
    raise RuntimeError(f"diabetes_model_v3.pkl saved as {model_size:.2f} MB — file may be corrupt. Re-train and re-save.")

print(f"diabetes_model_v3.pkl   {model_size:.1f} MB")
print(f"scaler_v3.pkl           saved")
print(f"shap_explainer_v3.pkl   {shap_size:.1f} MB")
print(f"feature_names_v3.json   {len(feature_cols)} features")
print()
print(f"Accuracy : {accuracy*100:.2f}%")
print(f"AUC-ROC  : {auc_roc:.4f}")
print(f"CV AUC   : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

<a id='12'></a>
## 12. Prediction Test

Loads all saved artifacts from disk and runs a full prediction pipeline to verify everything works correctly.

In [ ]:
# Guard: validate saved files before loading
for fname in ['diabetes_model_v3.pkl', 'scaler_v3.pkl', 'shap_explainer_v3.pkl', 'feature_names_v3.json']:
    if not os.path.exists(fname):
        raise FileNotFoundError(f"{fname} not found. Run Section 11 (Save Artifacts) first.")
    if os.path.getsize(fname) < 100:
        raise RuntimeError(f"{fname} is empty or corrupt. Run Sections 8-11 from the beginning.")

loaded_model     = joblib.load('diabetes_model_v3.pkl')
loaded_scaler    = joblib.load('scaler_v3.pkl')
loaded_explainer = joblib.load('shap_explainer_v3.pkl')
with open('feature_names_v3.json') as f:
    loaded_features = json.load(f)

# Verify the loaded model is fitted
try:
    check_is_fitted(loaded_model)
except Exception:
    raise RuntimeError(
        "Loaded model is not fitted — the saved .pkl file is corrupt.\n"
        "Delete diabetes_model_v3.pkl, re-run Sections 8 and 11, then retry."
    )

print(f"Model   : {type(loaded_model).__name__}")
print(f"Scaler  : {type(loaded_scaler).__name__}")
print(f"Features: {len(loaded_features)}")
print("All artifacts loaded and validated.")

In [ ]:
# Dummy patient — replace zeros with actual patient values in production
dummy_patient = np.zeros((1, len(loaded_features)))
dummy_scaled  = loaded_scaler.transform(dummy_patient)

prediction = loaded_model.predict(dummy_scaled)[0]
proba      = loaded_model.predict_proba(dummy_scaled)[0]
label      = 'HIGH-RISK' if prediction == 1 else 'STABLE'

print(f"Prediction : {prediction} ({label})")
print(f"Stable     : {proba[0]:.4f}")
print(f"High-Risk  : {proba[1]:.4f}")

In [ ]:
shap_explanation = loaded_explainer(dummy_scaled)
shap_explanation.feature_names = loaded_features

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_explanation[0], max_display=12, show=False)
plt.title('SHAP Explanation — Prediction Breakdown', fontweight='bold', fontsize=12, pad=15)
plt.tight_layout()
plt.show()